In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import stats
import matplotlib.pyplot as plt
from scipy import sparse

In [ ]:
adata = sc.read_h5ad(
    "../../data/cell_marker_matrix/21-54221-8/21-54221-8_cell_marker_matrix.h5ad"
)
# QC
# Filter all zero marker cells
# 1. 找到所有 positive_fraction 列
threshold = 0.1

pos_frac_cols = [c for c in adata.obs.columns if c.endswith("_positive_fraction")]

adata_qc = adata[adata.obs[pos_frac_cols].fillna(0).ge(threshold).any(axis=1)].copy()
adata_qc
# adata_use = adata_qc.copy()
# adata_use.layers["X_before_zscore_mean_filter"] = adata_use.X.copy()

# markers = (
#     adata_use.var["marker"].astype(str).tolist()
#     if "marker" in adata_use.var.columns
#     else adata_use.var_names.astype(str).tolist()
# )

# zmean_cols = [f"{m}_log1p_zscore_mean" for m in markers]

# missing_cols = [c for c in zmean_cols if c not in adata_use.obs.columns]
# if missing_cols:
#     raise ValueError(f"Missing columns in adata.obs: {missing_cols}")

# zmean_mat = adata_use.obs[zmean_cols].fillna(0).to_numpy()
# zero_mask = np.isclose(zmean_mat, 0)

# if sparse.issparse(adata_use.X):
#     X = adata_use.X.tocsr(copy=True)
#     X = X.multiply((~zero_mask).astype(X.dtype))
#     X.eliminate_zeros()
#     adata_use.X = X
# else:
#     X = np.asarray(adata_use.X).copy()
#     X[zero_mask] = 0
#     adata_use.X = X

# print("Finished.")
# print("Masked cell-marker positions:", zero_mask.sum())
# print("Masked fraction:", zero_mask.mean())
# print(adata_use)

In [ ]:
ccl19_col = "CCL19_log1p_zscore_mean"
ccr7_col = "CCR7_log1p_zscore_mean"

df = adata_use.obs[[ccl19_col, ccr7_col]].copy()
df = df.fillna(0)

df.columns = ["CCL19", "CCR7"]

print(df.describe())

In [ ]:
rho, pval = stats.spearmanr(df["CCL19"], df["CCR7"])

print("Cell-level Spearman correlation")
print("rho =", rho)
print("p-value =", pval)

In [ ]:
r, pval_pearson = stats.pearsonr(df["CCL19"], df["CCR7"])

print("Cell-level Pearson correlation")
print("r =", r)
print("p-value =", pval_pearson)

In [ ]:
mask_union = (df["CCL19"] > 0) | (df["CCR7"] > 0)

df_union = df.loc[mask_union].copy()

rho_union, pval_union = stats.spearmanr(df_union["CCL19"], df_union["CCR7"])

print("Union-positive Spearman correlation")
print("n cells =", df_union.shape[0])
print("rho =", rho_union)
print("p-value =", pval_union)

In [ ]:
plt.figure(figsize=(6, 5))
mask_union = (df["CCL19"] > 0) | (df["CCR7"] > 0)

df_union = df.loc[mask_union].copy()
plt.hexbin(df_union["CCL19"], df_union["CCR7"], gridsize=80, bins="log", mincnt=1)

plt.xlabel("CCL19")
plt.ylabel("CCR7")
plt.title(f"CCL19 vs CCR7 cell-level correlation\nSpearman rho={rho:.3f}")
plt.colorbar(label="log10(cell count)")
plt.tight_layout()
plt.show()

In [ ]:
# Spatial grid-level Spearman correlation
xy = adata_qc.obsm["spatial"]
x = xy[:, 0]
y = xy[:, 1]

mpp = 0.278
grid_size = int(100 / mpp)

obs = adata_qc.obs.copy()
obs["x"] = x
obs["y"] = y

obs["grid_x"] = np.floor(obs["x"] / grid_size).astype(int)
obs["grid_y"] = np.floor(obs["y"] / grid_size).astype(int)
obs["grid_id"] = obs["grid_x"].astype(str) + "_" + obs["grid_y"].astype(str)

grid_df = obs.groupby("grid_id").agg(
    CCL19_mean=("CCL19_log1p_zscore_mean", "mean"),
    CCR7_mean=("CCR7_log1p_zscore_mean", "mean"),
    CCL19_pos_frac=("CCL19_positive_fraction", "mean"),
    CCR7_pos_frac=("CCR7_positive_fraction", "mean"),
    n_cells=("cell_id", "count"),
    x=("x", "mean"),
    y=("y", "mean"),
)

# 过滤掉细胞太少的 grid
grid_df = grid_df[grid_df["n_cells"] >= 10].copy()

rho_grid, p_grid = stats.spearmanr(grid_df["CCL19_mean"], grid_df["CCR7_mean"])

print("Spatial grid-level Spearman correlation")
print("n grids =", grid_df.shape[0])
print("rho =", rho_grid)
print("p-value =", p_grid)

In [ ]:
plt.figure(figsize=(6, 5))

plt.scatter(grid_df["CCL19_mean"], grid_df["CCR7_mean"], s=10, alpha=0.5)

plt.xlabel("Grid mean CCL19")
plt.ylabel("Grid mean CCR7")
plt.title(f"Grid-level CCL19 vs CCR7\nSpearman rho={rho_grid:.3f}")
plt.tight_layout()
plt.show()